In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.spatial.distance import cdist
from ipywidgets import IntSlider, FloatSlider, Dropdown, VBox, HBox, interactive_output
from IPython.display import display
from pykrige.ok import OrdinaryKriging

# ----------------------------
# Champ "vrai" synthétique
# ----------------------------
def true_field(x, y):
    return (
        1.2 * np.exp(-((x - 25)**2 + (y - 30)**2) / 180)
        + 0.9 * np.exp(-((x - 70)**2 + (y - 65)**2) / 250)
        - 0.7 * np.exp(-((x - 55)**2 + (y - 25)**2) / 120)
        + 0.25 * np.sin(x / 8)
        + 0.15 * np.cos(y / 10)
    )

# ----------------------------
# Estimateur polygones d'influence
# = plus proche voisin
# ----------------------------
def polygon_influence_estimate(x_obs, y_obs, z_obs, xg, yg):
    grid_points = np.column_stack([xg.ravel(), yg.ravel()])
    obs_points = np.column_stack([x_obs, y_obs])
    d = cdist(grid_points, obs_points)
    idx = np.argmin(d, axis=1)
    return z_obs[idx].reshape(xg.shape)

# ----------------------------
# Démo
# ----------------------------
def demo_compare(
    n_samples=12,
    seed=0,
    noise=0.10,
    grid_size=50,
    variogram_model="spherical",
    nlags=8
):
    rng = np.random.default_rng(seed)

    # Domaine
    xmin, xmax = 0, 100
    ymin, ymax = 0, 100

    # Grille
    gx = np.linspace(xmin, xmax, grid_size)
    gy = np.linspace(ymin, ymax, grid_size)
    xg, yg = np.meshgrid(gx, gy)

    z_true = true_field(xg, yg)

    # Echantillons
    x_obs = rng.uniform(xmin + 5, xmax - 5, n_samples)
    y_obs = rng.uniform(ymin + 5, ymax - 5, n_samples)
    z_obs_true = true_field(x_obs, y_obs)
    z_obs = z_obs_true + rng.normal(0, noise, size=n_samples)

    # ----------------------------
    # PI / Voronoï
    # ----------------------------
    z_pi = polygon_influence_estimate(x_obs, y_obs, z_obs, xg, yg)

    # ----------------------------
    # Krigeage ordinaire
    # ----------------------------
    ok = OrdinaryKriging(
        x_obs, y_obs, z_obs,
        variogram_model=variogram_model,
        nlags=nlags,
        verbose=False,
        enable_plotting=False
    )

    z_ok, ss_ok = ok.execute("grid", gx, gy)
    z_ok = np.asarray(z_ok)

    # ----------------------------
    # Affichage
    # ----------------------------
    vmin = min(z_true.min(), z_pi.min(), z_ok.min())
    vmax = max(z_true.max(), z_pi.max(), z_ok.max())

    fig, axes = plt.subplots(2, 2, figsize=(12, 10))

    # Vrai champ
    im0 = axes[0, 0].imshow(
        z_true, origin="lower", extent=[xmin, xmax, ymin, ymax],
        vmin=vmin, vmax=vmax
    )
    axes[0, 0].scatter(x_obs, y_obs, c="k", s=25)
    axes[0, 0].set_title("Champ vrai")
    axes[0, 0].set_xlabel("X")
    axes[0, 0].set_ylabel("Y")

    # PI
    im1 = axes[0, 1].imshow(
        z_pi, origin="lower", extent=[xmin, xmax, ymin, ymax],
        vmin=vmin, vmax=vmax
    )
    axes[0, 1].scatter(x_obs, y_obs, c="k", s=25)
    axes[0, 1].set_title("Polygones d'influence")
    axes[0, 1].set_xlabel("X")
    axes[0, 1].set_ylabel("Y")

    # Krigeage
    im2 = axes[1, 0].imshow(
        z_ok, origin="lower", extent=[xmin, xmax, ymin, ymax],
        vmin=vmin, vmax=vmax
    )
    axes[1, 0].scatter(x_obs, y_obs, c="k", s=25)
    axes[1, 0].set_title("Krigeage ordinaire")
    axes[1, 0].set_xlabel("X")
    axes[1, 0].set_ylabel("Y")

    # Différence
    diff = z_ok - z_pi
    im3 = axes[1, 1].imshow(
        diff, origin="lower", extent=[xmin, xmax, ymin, ymax]
    )
    axes[1, 1].scatter(x_obs, y_obs, c="k", s=25)
    axes[1, 1].set_title("Différence : OK - PI")
    axes[1, 1].set_xlabel("X")
    axes[1, 1].set_ylabel("Y")

    # Barres de couleur
    cbar1 = fig.colorbar(im0, ax=[axes[0, 0], axes[0, 1], axes[1, 0]], shrink=0.8)
    cbar1.set_label("Valeur")
    cbar2 = fig.colorbar(im3, ax=axes[1, 1], shrink=0.8)
    cbar2.set_label("Différence")

    # Petit résumé
    mae_pi = np.mean(np.abs(z_pi - z_true))
    mae_ok = np.mean(np.abs(z_ok - z_true))

    fig.suptitle(
        f"Comparaison PI vs Krigeage | MAE PI = {mae_pi:.3f} | MAE OK = {mae_ok:.3f}",
        fontsize=14
    )

    plt.tight_layout()
    plt.show()

# ----------------------------
# Widgets
# ----------------------------
w_n = IntSlider(value=12, min=4, max=40, step=1, description="Nb points", continuous_update=False)
w_seed = IntSlider(value=0, min=0, max=50, step=1, description="Seed", continuous_update=False)
w_noise = FloatSlider(value=0.10, min=0.0, max=0.5, step=0.01, description="Bruit", continuous_update=False)
w_grid = IntSlider(value=50, min=20, max=80, step=5, description="Grille", continuous_update=False)
w_vario = Dropdown(
    options=["linear", "power", "gaussian", "spherical", "exponential"],
    value="spherical",
    description="Variogramme"
)
w_nlags = IntSlider(value=8, min=4, max=20, step=1, description="nlags", continuous_update=False)

out = interactive_output(
    demo_compare,
    {
        "n_samples": w_n,
        "seed": w_seed,
        "noise": w_noise,
        "grid_size": w_grid,
        "variogram_model": w_vario,
        "nlags": w_nlags
    }
)

ui = VBox([
    HBox([w_n, w_seed, w_noise]),
    HBox([w_grid, w_vario, w_nlags]),
    out
])

display(ui)